<details>
<summary><b>Copyright (C) 2025 Advanced Micro Devices, Inc. License: BSD-3-Clause</b></summary>

<pre>
Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved.

SPDX-License-Identifier: BSD-3-Clause

Portions of this notebook consist of AI-generated content. AI-assisted
content has been reviewed and validated by the authors.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice,
   this list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its contributors
   may be used to endorse or promote products derived from this software
   without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE
ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE
LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR
CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF
SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS
INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN
CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE)
ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE
POSSIBILITY OF SUCH DAMAGE.
</pre>

</details>

# Always-On Physical AI with AMD Ryzen AI Software

This notebook demonstrates how to run real-time vision models on the AMD Ryzen AI Neural Processing Unit (NPU). Leveraging the NPU for inference can accelerate performance while offloading work from the CPU and GPU. We start with a single pose model, compare CPU and NPU performance, go live on a webcam, and finish with a robot arm simulation in which your hand drives a simulated robot arm. Every model runs on the NPU, and nothing goes to the cloud.

The core skill this notebook teaches is straightforward: port an ONNX model to the NPU, then confirm it landed there. Everything else builds on that step. In the sections that follow, we'll walk through:

1. **Get the model**: load the YOLO26 pose weights (downloaded on first run).
2. **Export to ONNX**: the format ONNX Runtime and the NPU execution provider (VitisAI EP) consume.
3. **Compile for the NPU**: VAIML (the Ryzen AI compiler) converts the FP32 graph into BF16 code for the NPU.
4. **Build the inference pipeline**: the preprocess, run, and postprocess steps shared by the CPU and NPU.
5. **Run on CPU and NPU**: run the same model on the CPU and the NPU, switching only the provider.
6. **Compare CPU and NPU**: latency, throughput, CPU load, and power, side by side.
7. **Add a second model on the NPU**: a 21-point hand-landmark network running alongside YOLO26 pose. Pose finds a raised wrist, and the hand model reads the fingers in that crop.
8. **Watch it live**: pose and hand together on your webcam, both on the NPU.
9. **Robot arm simulation**: your hand drives a simulated SO-101 arm to pick and place a cube, with both models on the NPU.

### From ONNX to the NPU

The one thing you actually choose is the **execution provider** you pass to ONNX Runtime. The same model and the same `InferenceSession` API run on either engine: `providers=["VitisAIExecutionProvider"]` lands the model on the NPU, `providers=["CPUExecutionProvider"]` keeps it on the CPU. If you *ask* for the NPU but the NPU's execution provider (VitisAI EP) can't take the model, ONNX Runtime falls back to the CPU — it looks identical but is much slower, which is why you always confirm where it landed.

The NPU path also takes `provider_options` (the VAIML config file plus the compile cache); the CPU path needs none of that and runs the graph in FP32.

```mermaid
flowchart LR
    onnx["ONNX model<br/><small>FP32 graph</small>"]
    ort["onnxruntime<br/><small>InferenceSession</small>"]
    npu["NPU<br/><small>BF16 via VAIML</small>"]
    cpu["CPU<br/><small>FP32</small>"]
    opts["provider_options<br/><small>config_file · cache_dir/cache_key · target VAIML</small>"]

    onnx --> ort
    ort -->|"NPU execution provider (VitisAI EP)"| npu
    ort -->|"CPU EP"| cpu
    ort -.->|"fallback"| cpu
    opts -.->|"configures (NPU only)"| npu

    classDef npu fill:#e8f0ff,stroke:#4f78d0,stroke-width:2px;
    classDef fb fill:#f4f4f4,stroke:#9aa4b0,stroke-width:1px,color:#555;
    class npu npu;
    class cpu fb;
    class opts fb;
```

### 0. Imports and paths

Everything is imported up front so later cells stay focused on the workflow. The central packages: `onnxruntime` runs the model on the CPU or the NPU, `ultralytics.YOLO` loads the checkpoint and exports it to ONNX, and `ipywidgets` drives the live video and dashboard. The rest is the standard scientific stack (`numpy`, `OpenCV`, `matplotlib`).

In [ ]:
import os, sys, time, json, shutil, warnings

# Ultralytics: no auto-install, no network access.
os.environ["YOLO_AUTOINSTALL"] = "false"
os.environ["YOLO_OFFLINE"] = "true"
os.environ.setdefault("WORKSHOP_OFFLINE", "true")

from pathlib import Path
from collections import deque

import numpy as np
import cv2
import onnx
import onnxruntime
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from ultralytics import YOLO

warnings.filterwarnings("ignore")
onnxruntime.set_default_logger_severity(3)  # errors only

`workshop_utils` bundles the helper modules that keep the cells focused on the workflow: session building (`ep_factory`), preprocessing and drawing (`yolo_pipeline`, `pose_common`), the compiler-split and benchmark plots (`partition_viz`, `benchmark_viz`), the live resource dashboard (`resource_monitor`, `live_dashboard`), the threaded webcam demo (`threaded_inference`), and the SO-101 inverse kinematics (`arm_ik`). Each is introduced where it is first used, so there is nothing to memorize here.

In [ ]:
# Resolve repo paths whether this notebook is opened from the repo root or notebook/.
HERE = Path.cwd()
NB_DIR = HERE / "notebook" if (HERE / "notebook" / "workshop_utils").is_dir() else HERE
REPO = NB_DIR.parent
sys.path.insert(0, str(NB_DIR))

from workshop_utils.ep_factory import npu_session_options, redirect_native_output
from workshop_utils.yolo_pipeline import preprocess_frame
from workshop_utils.pose_common import (
    draw_poses,
)  # boxes + skeleton + keypoints overlay (ultralytics palette)
from workshop_utils.resource_monitor import ResourceMonitor, avg_load_during
from workshop_utils.live_dashboard import LiveDashboard
from workshop_utils.partition_viz import plot_partition
from workshop_utils.benchmark_viz import plot_cpu_vs_npu
from workshop_utils.threaded_inference import run_webcam_demo, make_live_pose_controls

This cell sets the paths used throughout the notebook. The three that matter: `ONNX_PATH` (the exported model), and `CACHE_DIR` / `CACHE_KEY` (the NPU execution provider (VitisAI EP) compile cache: the first run compiles and writes there, and every run after loads it in about a second).

In [ ]:
MODEL_NAME = "yolo26s-pose"
PT_PATH = NB_DIR / f"{MODEL_NAME}.pt"
ONNX_PATH = NB_DIR / f"{MODEL_NAME}.onnx"
CACHE_DIR = REPO / "cache"
CACHE_KEY = f"{MODEL_NAME}_fp32"
SAMPLE_VIDEO = REPO / "assets" / "sample_video.mp4"
SAMPLE_IMAGE = REPO / "assets" / "sample.jpg"

print(f"Model      : {MODEL_NAME}")
print(f"ONNX model : {ONNX_PATH}  exists={ONNX_PATH.exists()}")
print(
    f"Cache dir  : {CACHE_DIR / CACHE_KEY}  exists={(CACHE_DIR / CACHE_KEY).exists()}"
)

### Check the NPU is visible
Before anything else, confirm ONNX Runtime can see the NPU execution provider (VitisAI EP). If `VitisAIExecutionProvider` is missing, the Ryzen AI environment is not active.

In [ ]:
print(f"ORT providers: {onnxruntime.get_available_providers()}")

### 1. Get the model

YOLO26 is a recent Ultralytics detection model. The pose task adds a 17-keypoint head on top of the detection backbone. This notebook uses the YOLO26 pose model for a good balance of accuracy and speed on the NPU. This cell loads the weights (`yolo26s-pose.pt`); Ultralytics fetches them on the first run if they are not already present, and loads them from disk on every run after.

In [ ]:
model = YOLO(
    str(PT_PATH)
)  # anchored to NB_DIR (the local .pt); no accidental GitHub download regardless of launch cwd
print(model.info())

To confirm the model loads and runs, we start with the CPU: a single forward pass on a sample image.

In [ ]:
_result = model.predict(str(SAMPLE_IMAGE), conf=0.4, verbose=False)[0]
_annotated = _result.plot()  # BGR numpy array with boxes + keypoints drawn

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(cv2.cvtColor(_annotated, cv2.COLOR_BGR2RGB))
ax.axis("off")
ax.set_title(f"{MODEL_NAME}: CPU inference on sample image", fontsize=10)
plt.tight_layout()
plt.show()

print(
    f"Detected {len(_result.boxes)} person(s)  |  {len(_result.keypoints.data)} pose(s)"
)

### 2. Export to ONNX

**ONNX** (Open Neural Network Exchange) is the interchange format that ONNX Runtime and the NPU execution provider (VitisAI EP) read. Here we export the model at opset 17, batch size 1, and a fixed 640x640 input.

```mermaid
flowchart LR
    pt["yolo26s-pose.pt<br/>(PyTorch)"] --> exp["export<br/>opset 17, static 640x640, FP32"]
    exp --> onnx["yolo26s-pose.onnx<br/>(FP32)"]
    onnx --> ep["compiled by the VitisAI EP<br/>(Section 3)"]
```

One `model.export()` call turns the PyTorch weights into the static-shape ONNX
graph the NPU execution provider (VitisAI EP) compiles onto the NPU next.

In [ ]:
if not ONNX_PATH.exists():
    exported = model.export(
        format="onnx",
        imgsz=640,  # Export at 640x640 image size; smaller size is faster but less accurate
        opset=17,  # ONNX operator-set version; 17 is well-supported by ONNX Runtime / VAIML
        simplify=True,  # run onnx-simplifier to fold constants + drop redundant nodes (cleaner graph to compile)
        dynamic=False,  # static input shape; VAIML compiler needs fixed shapes to generate AIE code
        batch=1,
        nms=False,  # select YOLO26's NMS-free one-to-one head -> [1, 300, 57]; omitting this exports the raw grid [1, 56, 8400]
    )

    # Ultralytics writes the .onnx next to the .pt; move it to ONNX_PATH.
    exported = Path(exported)
    if exported.resolve() != ONNX_PATH.resolve():
        ONNX_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(exported), str(ONNX_PATH))
    print(f"ONNX model written to: {ONNX_PATH}  exists={ONNX_PATH.exists()}")
else:
    print(f"Using existing ONNX: {ONNX_PATH}")

m = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(m)
print(
    "Inputs: ",
    {
        i.name: [d.dim_value for d in i.type.tensor_type.shape.dim]
        for i in m.graph.input
    },
)
print(
    "Outputs:",
    {
        o.name: [d.dim_value for d in o.type.tensor_type.shape.dim]
        for o in m.graph.output
    },
)

Normally a detector outputs many overlapping boxes that a cleanup step called non-maximum suppression (NMS) filters down. YOLO26 is NMS-free: it uses one-to-one label assignment during training, so it produces non-overlapping predictions directly. The output is a `[1, 300, 57]` tensor of 300 pre-filtered detections. Each detection carries 4 box coordinates, 1 confidence score, and 17 keypoints of (x, y, visibility). There is no separate non-maximum-suppression step.

### 3. Compile for the NPU

This section has two steps across two cells. First, we write a config for **VAIML**, the Ryzen AI compiler inside the NPU execution provider (VitisAI EP). The key flag is `enable_f32_to_bf16_conversion`, which converts each FP32 layer to **BF16**, the datatype the NPU's **AIE** (AI Engine) cores compute in. This replaces a separate quantization step.

Writing the config does not compile anything on its own. The compile runs when you build the `InferenceSession` in the next cell, so we write the config first.

Under the hood, building the NPU session runs the **VAIML** compiler, which turns your plain FP32 ONNX graph into a cached NPU binary:

```mermaid
flowchart LR
    onnx["FP32 ONNX<br/><small>your exported graph</small>"]
    subgraph vaiml ["VAIML compiler (VitisAI EP)"]
        direction TB
        part["partition<br/><small>pick AIE-supported subgraphs</small>"]
        bf16["FP32 to BF16<br/><small>AIE compute datatype</small>"]
        codegen["AIE code-gen<br/><small>schedule onto the tiles</small>"]
        part --> bf16 --> codegen
    end
    cache["cached binary<br/><small>cache_dir/cache_key/*.rai</small>"]
    load["session ready<br/><small>~1s load on every later run</small>"]

    onnx --> vaiml --> cache --> load

    classDef hw fill:#e8f0ff,stroke:#4f78d0,stroke-width:2px;
    classDef dim fill:#f4f4f4,stroke:#9aa4b0,stroke-width:1px,color:#555;
    class cache hw;
    class load hw;
    class onnx dim;
```

The first compile takes tens of minutes and is done once (it is already cached for this workshop). Every later run just loads the cached binary — the ~1s "cache hit" you see below.

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
config_path = CACHE_DIR / "vitisai_config.json"

config_path.write_text(
    json.dumps(
        {
            "passes": [
                {"name": "init", "plugin": "vaip-pass_init"},
                {
                    "name": "vaiml_partition",
                    "plugin": "vaip-pass_vaiml_partition",
                    "vaiml_config": {
                        "enable_f32_to_bf16_conversion": True,  # cast FP32 -> BF16, the datatype the AIE (AI Engine) cores compute in (takes effect at compile time)
                        "keep_outputs": True,
                        "logging_level": "info",
                    },
                },
            ],
            "target": "VAIML",
            "targets": [{"name": "VAIML", "pass": ["init", "vaiml_partition"]}],
        },
        indent=2,
    )
)

print("Config written to", config_path)

Building the session triggers the compile. It is an ordinary `onnxruntime.InferenceSession`. The `providers=["VitisAIExecutionProvider"]` argument is the switch that puts the model on the NPU, and the `provider_options` argument points VAIML at the config and the cache. This is the same ONNX Runtime API you use for the CPU: you select the NPU without rewriting your model or your inference code.

VAIML partitions the graph, selects which subgraphs run on the AIE cores, generates BF16 code, and writes the result to `cache_dir/cache_key/`. The first run compiles (30 to 60 minutes). Every run after that loads from the cache in about a second.

In [ ]:
print("Building NPU session (cache hit → ~1s, first compile → ~30-60 min)...")
# For first compile, send native compiler lines to a log file (cache/<key>/compile.log).
with redirect_native_output(CACHE_DIR / CACHE_KEY / "compile.log"):
    npu_session = onnxruntime.InferenceSession(
        str(ONNX_PATH),
        sess_options=npu_session_options(),
        providers=[
            "VitisAIExecutionProvider"
        ],  # this is what runs the model on the NPU (vs "CPUExecutionProvider")
        provider_options=[
            {
                "config_file": str(config_path),
                "cache_dir": str(CACHE_DIR),
                "cache_key": CACHE_KEY,
                "target": "VAIML",
            }
        ],
    )
print("Ready.")

After compilation, VAIML writes a pass summary and two CSV files (`graph_partition_trace.csv` and `gops.csv`) that map each operator to its device and compute cost. The operators that fall back to the CPU are small bookkeeping operators in the detection head (`TopK`, `GatherElements`, `Mod`, and similar). There are several of them, but they perform almost no compute. Measured by GOPs, the split is close to 100% NPU.

In [ ]:
summary = CACHE_DIR / CACHE_KEY / "final-vaiml-pass-summary.txt"
if summary.exists():
    print(summary.read_text())

plot_partition(CACHE_DIR, CACHE_KEY)

### 4. Build the inference pipeline

We define one pipeline that runs on both the CPU and the NPU: decode the model output, draw the skeleton, and loop over frames. Nothing runs the model yet: that happens in Section 5.

#### Postprocess

YOLO26's output is already clean boxes and keypoints, so the postprocessing is short: drop low-confidence detections, then undo the letterbox padding.

The raw output is `[1, 300, 57]`. Each of the 300 rows is a candidate detection, laid out across the 57 columns as follows:

| Columns | Contents |
|---|---|
| 0-3 | bounding box (x1, y1, x2, y2) in the letterboxed 640x640 space |
| 4 | confidence score |
| 5 | class id (always 0, the single "person" class, skipped in decoding) |
| 6-56 | 17 keypoints, 3 values each (x, y, visibility) |

`postprocess_pose` drops detections below the confidence threshold, then undoes the letterbox padding to map the coordinates back to the original image size.

In [ ]:
# Detection thresholds (used by postprocess_pose and the live pose controls).
# The COCO skeleton bone map and drawing live in workshop_utils.pose_common
# (draw_poses, imported in Section 0).
CONF_THRESHOLD = 0.4  # minimum box confidence to keep a detection
KPT_VIS_THRESHOLD = 0.3  # minimum keypoint visibility to draw a joint


def postprocess_pose(outputs, original_shape, info, conf_threshold=CONF_THRESHOLD):
    """[1, 300, 57] → (boxes_xyxy, scores, keypoints[N, 17, 3]) in original image coords."""
    preds = outputs[0][0]  # [300, 57]
    boxes = preds[:, :4].copy()
    scores = preds[:, 4]
    kpts = preds[:, 6:].reshape(-1, 17, 3).copy()

    mask = scores > conf_threshold
    boxes, scores, kpts = boxes[mask], scores[mask], kpts[mask]
    if len(boxes) == 0:
        return boxes, scores, kpts

    # Undo letterbox: subtract padding, then divide by scale ratio
    boxes[:, [0, 2]] -= info.pad_x
    boxes[:, [1, 3]] -= info.pad_y
    boxes /= info.ratio
    kpts[..., 0] -= info.pad_x
    kpts[..., 1] -= info.pad_y
    kpts[..., :2] /= info.ratio

    h, w = original_shape
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, w)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, h)
    return boxes, scores, kpts

#### Inference loop

`run_inference` runs the full pipeline (capture, preprocess, infer, postprocess, draw) and shows the frame live. The readout reports the end-to-end frame rate and latency. Section 5 reports the raw NPU compute time.

In [ ]:
def run_inference(session, source, max_frames=500, label=""):
    img_widget = widgets.Image(format="jpeg", width=640)
    fps_label = widgets.HTML()
    display(widgets.VBox([fps_label, img_widget]))

    input_name = session.get_inputs()[0].name
    if isinstance(source, int):
        cap = cv2.VideoCapture(source, cv2.CAP_V4L2)
        cap.set(
            cv2.CAP_PROP_BUFFERSIZE, 1
        )  # webcam: keep only the newest frame (low latency)
    else:
        cap = cv2.VideoCapture(source)

    e2e_times_ms, frame_count = [], 0
    try:
        while frame_count < max_frames:
            ok, frame = cap.read()
            if not ok:
                break

            nchw, info = preprocess_frame(
                frame
            )  # letterbox to the model's fixed 640x640 input
            t_infer = time.perf_counter()
            outputs = session.run(
                None, {input_name: nchw}
            )  # the inference - CPU or NPU, decided by the session
            infer_ms = (
                time.perf_counter() - t_infer
            ) * 1000  # inference compute time (CPU or NPU, per session)

            t0 = time.perf_counter()
            boxes, scores, kpts = postprocess_pose(outputs, frame.shape[:2], info)
            annotated = draw_poses(frame, boxes, scores, kpts)
            _, jpg = cv2.imencode(".jpg", annotated, [cv2.IMWRITE_JPEG_QUALITY, 80])
            img_widget.value = jpg.tobytes()
            e2e_ms = (
                infer_ms + (time.perf_counter() - t0) * 1000
            )  # end-to-end: inference + Python pre/post-processing

            e2e_times_ms.append(e2e_ms)
            frame_count += 1

            recent = e2e_times_ms[-10:]
            fps_ema = 1000 / (sum(recent) / len(recent))
            fps_label.value = (
                f"<b>{label}</b> &nbsp; "
                f"<span style='color:#0a8'>{fps_ema:.1f} fps</span> &nbsp; "
                f"({1000 / fps_ema:.1f} ms/frame) &nbsp; "
                f"{len(boxes)} people"
            )
    finally:
        cap.release()
    return e2e_times_ms

### 5. Run on CPU and NPU

Now we run the same pipeline on each engine. Only one thing changes: the session's provider decides whether `session.run()` lands on the CPU or the NPU. The model, the code, and the video are identical.

The run cell below is a **TRY ME**. As written, it runs on the CPU. To run it on the NPU, switch which line is commented and run it again. Each run is saved to `RESULTS`, and Section 6 compares them.

In [ ]:
# label -> (per-frame times ms, avg CPU load %, avg APU watts). Guard keeps earlier
# runs when you re-run this cell to swap engines.
if "RESULTS" not in globals():
    RESULTS = {}

# ============================== TRY ME ==============================
# Same model, same code - the ONE thing that changes is the Execution Provider
# (EP) you pass to ONNX Runtime. Comment one line, uncomment the other, re-run.
provider, label = "CPUExecutionProvider", "CPU FP32"  # <- active: runs on the CPU
# provider, label = "VitisAIExecutionProvider", "NPU BF16"  # <- uncomment: runs on the NPU/AIE
# ====================================================================

if provider == "VitisAIExecutionProvider":
    session = onnxruntime.InferenceSession(
        str(ONNX_PATH),
        sess_options=npu_session_options(),
        providers=["VitisAIExecutionProvider"],
        provider_options=[
            {
                "config_file": str(config_path),
                "cache_dir": str(CACHE_DIR),
                "cache_key": CACHE_KEY,
            }
        ],
    )
else:
    session = onnxruntime.InferenceSession(
        str(ONNX_PATH), providers=["CPUExecutionProvider"]
    )

times, load, watts = avg_load_during(
    lambda: run_inference(session, str(SAMPLE_VIDEO), max_frames=500, label=label)
)
RESULTS[label] = (times, load, watts)

p50 = np.percentile(
    times[20:], 50
)  # skip the first 20 frames (startup transients / AIE warm-up on NPU)
_w = f" | APU ~{watts:.0f} W" if watts else ""
print(f"{label}:  {1000/p50:.1f} fps  |  CPU ~{load:.0f}% busy{_w}")
print(
    "Saved. Now switch the engine above and run this cell again."
    if len(RESULTS) < 2
    else "Both engines recorded. Continue to Section 6."
)

### 6. Compare CPU and NPU

With both runs recorded, this section shows what the NPU provides. Frame rate is only part of the picture. The larger benefits are a freed CPU and lower power, which running the same model in FP32 on the CPU cannot deliver.

In [ ]:
# All the plotting lives in workshop_utils.benchmark_viz; the returned metrics
# keep p50_npu in the notebook globals so Section 8's dashboard can scale to it.
metrics = plot_cpu_vs_npu(
    RESULTS, cpu_key="CPU FP32", npu_key="NPU BF16", npu_warmup=20
)
if "p50_npu" in metrics:
    p50_npu = metrics["p50_npu"]

### 7. Add a second model on the NPU: hand landmarks

So far only YOLO26 pose has run on the NPU. The NPU is built to keep several always-on jobs alive at once, so here we add a **second** model, a 21-point **hand-landmark** network, running alongside pose on the same device.

The two models work as a team: pose finds the person and the **wrist** of a raised hand, we crop a small box around that wrist, and the hand model reads the 21 finger joints inside the crop. The second model only looks where the first already found a hand, which is cheap and stable. Section 9 reuses this same pose→wrist→hand hand-off; here we just draw the result.

The hand network is a MobileNet-style CNN that offloads 100% to the AIE cores (even cleaner than pose, whose detection head leaves a little bookkeeping on the CPU). The NPU holds two concurrent contexts, so pose + hand fills it exactly. The live demo below has a **toggle** to add and remove the hand model on the fly, and its per-frame time appears in the stat line next to the pose time.

In [ ]:
# Build the SECOND model, the 21-point hand-landmark net, on the NPU, using
# the same VAIML path as the pose model in Section 3. Section 9 reuses this exact
# same session, so we build it at most once: pose + hand = 2 resident NPU
# contexts, which is exactly the device budget.
from workshop_utils.hand_landmarks import DEFAULT_MODEL as HAND_ONNX, ensure_hand_model

ensure_hand_model(
    HAND_ONNX
)  # ~10 MB download on first run (timeout + progress + retries)
HAND_CACHE_KEY = "hand_landmark_fp32"

if (
    "npu_hand_session" in globals()
    and npu_hand_session.get_providers()[0] == "VitisAIExecutionProvider"
):
    print("NPU hand session already built; reusing it (still 2 NPU contexts).")
else:
    print("Building NPU hand session (cache hit → ~1s, first compile → ~3 min)...")
    # For first compile, send native compiler lines to a log file (cache/<key>/compile.log).
    with redirect_native_output(CACHE_DIR / HAND_CACHE_KEY / "compile.log"):
        npu_hand_session = onnxruntime.InferenceSession(
            str(HAND_ONNX),
            sess_options=npu_session_options(),  # same idle-CPU fix as the pose session
            providers=["VitisAIExecutionProvider"],
            provider_options=[
                {
                    "config_file": str(config_path),
                    "cache_dir": str(CACHE_DIR),
                    "cache_key": HAND_CACHE_KEY,
                    "target": "VAIML",
                }
            ],
        )
    print("Ready: pose + hand both on the NPU.")

print("hand provider:", npu_hand_session.get_providers()[0])

# Proof it's on the NPU: VAIML's own pass summary + the operator/compute split.
hand_summary = CACHE_DIR / HAND_CACHE_KEY / "final-vaiml-pass-summary.txt"
if hand_summary.exists():
    print(hand_summary.read_text())

plot_partition(CACHE_DIR, HAND_CACHE_KEY)

### 8. Watch it live: pose and hand, both on the NPU

The next cell goes live: the pose skeleton on your webcam, a Stop button, and the resource dashboard, with the second model wired in. Raise a hand and the hand model draws its 21 joints on the wrist pose found; the stat line shows both models' per-frame time. The **hand model** toggle adds and removes the second model live, so its time appears and disappears on the stat line, and the **NPU** gauge is fed by both models. Press **Stop** to end the demo cleanly.

**TRY ME:** while the demo runs, toggle the **hand model** on and off and move the confidence and keypoint sliders. Everything updates live.

In [ ]:
# Threaded live demo (Stop button + dashboard). Reuses npu_session (pose) +
# npu_hand_session (hand); the "hand model" toggle adds/removes the 2nd model live.

# ============================== TRY ME ==============================
# Swap which line is active - live webcam, or the bundled clip (decoded from disk
# with no fps cap, so the NPU runs flat-out):
SOURCE = 0  # live webcam (raise a hand)
# SOURCE = str(SAMPLE_VIDEO)             # the bundled clip
# ====================================================================

# Live tuning widgets (sliders + show/hide checkboxes) for the pose overlay.
# The hand-model on/off toggle is added automatically by run_webcam_demo when a
# hand_session is passed.
post_live, draw_live, control_widgets = make_live_pose_controls(
    postprocess_pose, draw_poses, conf=CONF_THRESHOLD, kpt_vis=KPT_VIS_THRESHOLD
)

demo = run_webcam_demo(
    npu_session,
    preprocess_frame,
    post_live,
    draw_live,
    source=SOURCE,
    label="NPU",
    hand_session=npu_hand_session,  # the SECOND model on the NPU
    hand_on=True,  # start with both models live
    extra_widgets=control_widgets,  # live pose sliders + show/hide
    npu_infer_ms=globals().get("p50_npu", 20.0),
)  # live dashboard under the feed

### 9. Robot arm simulation: pick and place with an SO-101

The same two models and NPU as the live demo, now driving a robot arm. Your hand flies the gripper, a pinch grabs the cube, and opening your hand drops it on the green target. The same command stream would drive a physical SO-101; here it is simulated on the iGPU.

| Engine | Job |
|--------|-----|
| **NPU** | YOLO26-pose finds you and crops a hand box; the 21-point hand model reads the fingers in that crop. Both nets run back to back. |
| **CPU** | Maps the 21 landmarks to a gripper target and grip state (pinch) with vector math and smoothing. No neural net; stays nearly idle. |
| **iGPU** | MuJoCo SO-101: inverse kinematics to the target, physics for the cube. |

Your hand drives two reliable axes: **left/right** and **up/down** (depth/reach is experimental, pass `enable_depth=True` to try it). The arm **holds its last position whenever tracking blinks**, so a dropped frame means "stay put" rather than a snap.

**How to play:** lower the gripper onto the cube and pinch to grab (it turns cyan, HUD shows CARRYING). Carry it over the green circle and open your fingers to drop; a good place bumps the counter and moves the target. **Reset cube** restarts. The **TRY ME** in the cell also swaps both nets onto the CPU to compare.

In [ ]:
# Reuse the Section 7 hand session if it exists; build it here only if you jumped
# straight to Section 9 (never a 3rd NPU context).
from workshop_utils.hand_landmarks import DEFAULT_MODEL as HAND_ONNX, ensure_hand_model
from workshop_utils.arm_teleop_mirror import run_arm_teleop_in_notebook

HAND_CACHE_KEY = "hand_landmark_fp32"

if (
    "npu_hand_session" in globals()
    and npu_hand_session.get_providers()[0] == "VitisAIExecutionProvider"
):
    print("Reusing the NPU hand session from Section 7: pose + hand = 2 NPU contexts.")
else:
    ensure_hand_model(
        HAND_ONNX
    )  # ~10 MB download on first run (timeout + progress + retries)
    print(
        "Section 7 not run; building the NPU hand session now "
        "(cache hit -> ~1s, first compile -> ~3 min)..."
    )
    # For first compile, send native compiler lines to a log file (cache/<key>/compile.log).
    with redirect_native_output(CACHE_DIR / HAND_CACHE_KEY / "compile.log"):
        npu_hand_session = onnxruntime.InferenceSession(
            str(HAND_ONNX),
            sess_options=npu_session_options(),  # same idle-CPU fix as the pose session
            providers=["VitisAIExecutionProvider"],
            provider_options=[
                {
                    "config_file": str(config_path),
                    "cache_dir": str(CACHE_DIR),
                    "cache_key": HAND_CACHE_KEY,
                    "target": "VAIML",
                }
            ],
        )
    print("Ready: pose + hand both on the NPU.")

# ============================== TRY ME ==============================
# The Execution Provider (EP) the teleop's two nets (pose + hand) run on.
# Comment one line, uncomment the other, re-run.
provider = "VitisAIExecutionProvider"  # <- active: both nets on the NPU/AIE
# provider = "CPUExecutionProvider"     # <- uncomment: both nets on the CPU
# ====================================================================

if provider == "VitisAIExecutionProvider":
    # Reuse the NPU sessions already built (Section 3 pose + the hand session
    # above): 2 of 2 NPU contexts, nothing rebuilt.
    pose_sess, hand_sess = npu_session, npu_hand_session
else:
    # Build a fresh CPU pose session; hand_sess=None -> teleop runs the hand on CPU too.
    pose_sess = onnxruntime.InferenceSession(
        str(ONNX_PATH), providers=["CPUExecutionProvider"]
    )
    hand_sess = None

# Your hand flies the gripper on two reliable image-plane axes: left/right + up/down.
# Depth/reach is experimental and OFF by default; pass enable_depth=True to try the
# in/out reach axis (hand toward/away from the camera).
TELEOP_VIDEO = REPO / "assets" / "teleop_sample.mp4"

# ============================== TRY ME ==============================
# Swap which line is active - live webcam, or the recorded clip:
# SOURCE = str(TELEOP_VIDEO)                 # the recorded clip (assets/teleop_sample.mp4)
SOURCE = 0  # live webcam (raise a hand and pinch)
# ====================================================================
teleop = run_arm_teleop_in_notebook(
    pose_sess,
    source=SOURCE,
    hand_session=hand_sess,
    feed_npu_gauge=(provider == "VitisAIExecutionProvider"),
    npu_infer_ms=globals().get("p50_npu", 20.0),
)  # live dashboard shows under the demo